# Mission 1 — 신고자 성별 분류 (Mel-spectrogram + CNN)

## 전체 파이프라인
```
오디오(WAV) → [startAt, endAt]로 구간 추출 → Mel-spectrogram (2D 이미지) → CNN 분류 → 남(0) / 여(1)
```

## JSON 구조 (확인된 형식)
```json
{
    "utterances": [
        {"id": "...", "startAt": 41184, "endAt": 41574, "text": "예.", "speaker": 1},
        ...
    ],
    "gender": "M",          ← 신고자 성별 레이블 (파일 최상위)
    "mediaType": "Mobile",
    ...
}
```

## 핵심 제약 사항
| 단계 | 사용 가능한 필드 |
|------|------------------|
| 학습 | startAt, endAt, speaker → gender(최상위)로 레이블 |
| 추론 | startAt, endAt **만** 사용 가능 |

## 주의: startAt / endAt 단위
- 값이 **밀리초(ms)** 형태 (예: 41184 ms = 41.184 초)
- librosa는 초(s) 단위를 받으므로 `/1000` 변환 필수

## 전처리 전략 (힌트 반영)
- **짧은 음성** (< TARGET_SEC): 오른쪽에 zero-padding
- **긴 음성** (> TARGET_SEC): 중앙 구간 crop

## 1. 환경 설정

In [ ]:
!pip install librosa soundfile -q
print('설치 완료')

In [ ]:
import os
import json
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from tqdm.notebook import tqdm

# 재현성 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')
if device.type == 'cuda':
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')

## 2. Google Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── 작업 폴더 (결과물 저장 위치) ──────────────────────────────────────────
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/2.신주아'
os.makedirs(WORK_DIR, exist_ok=True)

# ── 저장 파일 경로 (런타임 재시작 후에도 유지) ──────────────────────────
TRAIN_CSV       = f'{WORK_DIR}/train_labels.csv'
VAL_CSV         = f'{WORK_DIR}/val_labels.csv'
MODEL_PATH      = f'{WORK_DIR}/best_gender_cnn.pth'     # 최고 성능 모델
CHECKPOINT_PATH = f'{WORK_DIR}/checkpoint.pth'          # 이어서 학습용 체크포인트

# ── 데이터 원천 경로 ──────────────────────────────────────────────────────
BASE = '/content/drive/MyDrive/대학부 데이터'

PATHS = {
    'train_wav' : f'{BASE}/Training/1.원천데이터/TS_서울_구급',
    'train_json': f'{BASE}/Training/2.라벨링데이터/TS_서울_구급',
    'val_wav'   : f'{BASE}/Validation/1.원천데이터/TS_서울_구급',
    'val_json'  : f'{BASE}/Validation/2.라벨링데이터/TS_서울_구급',
}

print(f'작업 폴더: {WORK_DIR}')
print('\n데이터 경로 확인:')
for name, path in PATHS.items():
    exists = os.path.exists(path)
    count  = len(os.listdir(path)) if exists else 0
    print(f'  {name:12s}: {"OK" if exists else "NOT FOUND":10s} ({count} 파일)')

## 3. 데이터 탐색

In [ ]:
def find_files(directory, ext):
    """하위 폴더까지 재귀적으로 파일 검색"""
    return sorted(
        glob.glob(os.path.join(directory, f'*.{ext}')) +
        glob.glob(os.path.join(directory, f'**/*.{ext}'), recursive=True)
    )

train_wavs  = find_files(PATHS['train_wav'],  'wav')
train_jsons = find_files(PATHS['train_json'], 'json')
val_wavs    = find_files(PATHS['val_wav'],    'wav')
val_jsons   = find_files(PATHS['val_json'],   'json')

print(f'Train  WAV : {len(train_wavs):>5}개')
print(f'Train  JSON: {len(train_jsons):>5}개')
print(f'Val    WAV : {len(val_wavs):>5}개')
print(f'Val    JSON: {len(val_jsons):>5}개')

if train_wavs:  print(f'\nWAV  파일명 예시: {os.path.basename(train_wavs[0])}')
if train_jsons: print(f'JSON 파일명 예시: {os.path.basename(train_jsons[0])}')

In [ ]:
# JSON 최상위 구조 확인
with open(train_jsons[0], 'r', encoding='utf-8') as f:
    sample = json.load(f)

print('=== JSON 최상위 키 및 타입 ===')
for key, val in sample.items():
    if isinstance(val, list):
        inner = type(val[0]).__name__ if val else 'empty'
        print(f'  "{key}": list[{inner}]  (len={len(val)})')
        if val and isinstance(val[0], dict):
            print(f'    → 첫 항목 키: {list(val[0].keys())}')
            # startAt/endAt 값 확인 (ms인지 s인지)
            if 'startAt' in val[0]:
                print(f'    → startAt 예시값: {val[0]["startAt"]}  (ms 추정: {val[0]["startAt"]/1000:.3f}s)')
    elif isinstance(val, dict):
        print(f'  "{key}": dict')
    else:
        print(f'  "{key}": {type(val).__name__} = {repr(val)}')

## 4. 라벨 데이터 파싱

### 확인된 JSON 구조
- **`gender`**: 최상위 키 → 통화 전체의 신고자 성별 (`"M"` / `"F"`)
- **`startAt` / `endAt`**: **밀리초(ms)** 단위 → 파싱 시 `/1000` 으로 초 변환
- **utterance 리스트**: 키 이름이 데이터마다 다를 수 있어 자동 탐지
- **`speaker`**: 정수형 발화자 번호

### 레이블 전략
한 JSON 파일 안의 **모든 발화**에 파일 최상위 `gender`를 레이블로 부여합니다.  
(학습 시 `speaker` 필드를 이용해 신고자 발화만 필터링하면 노이즈 감소 가능)

In [ ]:
def normalize_gender(g):
    """gender 표기를 0(남) / 1(여) / -1(알 수 없음)으로 통일"""
    s = str(g).strip().lower()
    if s in ['남', '남성', 'm', 'male', '0']:
        return 0
    if s in ['여', '여성', 'f', 'female', '1']:
        return 1
    return -1


def find_wav(base_dir, stem):
    """파일 stem(확장자 없는 이름)으로 WAV 파일 탐색"""
    direct = os.path.join(base_dir, stem + '.wav')
    if os.path.exists(direct):
        return direct
    found = glob.glob(os.path.join(base_dir, '**', stem + '.wav'), recursive=True)
    return found[0] if found else None


def get_utterances(data):
    """
    dict에서 utterance 리스트를 자동 탐지.
    startAt 또는 start 키를 가진 dict 리스트를 찾아 반환.
    """
    for val in data.values():
        if (
            isinstance(val, list) and val
            and isinstance(val[0], dict)
            and ('startAt' in val[0] or 'start' in val[0])
        ):
            return val
    return []


def parse_labels(json_dir, wav_dir, min_duration_sec=0.5):
    """
    JSON 라벨 파일 파싱 → DataFrame 반환
    컬럼: wav_path, start(s), end(s), speaker, gender, duration(s)

    [확인된 JSON 구조]
    - gender  : 최상위 키, 파일 전체의 신고자 성별
    - startAt : 밀리초(ms) → /1000 으로 초 변환
    - speaker : 정수형 발화자 번호
    """
    records = []

    for jpath in find_files(json_dir, 'json'):
        stem = os.path.splitext(os.path.basename(jpath))[0]
        wav_path = find_wav(wav_dir, stem)
        if wav_path is None:
            continue

        with open(jpath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if not isinstance(data, dict):
            continue

        # 신고자 성별 (최상위 레벨)
        gender = normalize_gender(data.get('gender', ''))
        if gender < 0:
            continue

        # utterance 리스트 자동 탐지
        utterances = get_utterances(data)

        for utt in utterances:
            # startAt/endAt 는 밀리초 → 초 변환
            start_sec = float(utt.get('startAt', utt.get('start', 0))) / 1000.0
            end_sec   = float(utt.get('endAt',   utt.get('end',   0))) / 1000.0
            duration  = end_sec - start_sec

            if duration >= min_duration_sec:
                records.append({
                    'wav_path': wav_path,
                    'start'   : round(start_sec, 4),
                    'end'     : round(end_sec,   4),
                    'speaker' : int(utt.get('speaker', -1)),
                    'gender'  : gender,
                    'duration': round(duration, 3),
                })

    return pd.DataFrame(records)

In [ ]:
# CSV가 이미 있으면 바로 불러오고, 없으면 파싱 후 저장
if os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV):
    print('✅ 저장된 CSV 불러오는 중... (파싱 생략)')
    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    print(f'   Train: {len(train_df):,}개  |  Val: {len(val_df):,}개')

else:
    print('CSV 없음 → 파싱 시작...')
    print('Training 데이터 파싱 중...')
    train_df = parse_labels(PATHS['train_json'], PATHS['train_wav'])
    print('Validation 데이터 파싱 중...')
    val_df   = parse_labels(PATHS['val_json'], PATHS['val_wav'])

    # Drive에 저장 (다음 실행부터 파싱 생략)
    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV,     index=False)
    print(f'✅ CSV 저장 완료 → {WORK_DIR}')

# 결과 확인
print(f'\nTrain: {len(train_df):,} 샘플')
print(f'Val  : {len(val_df):,} 샘플')

print('\n── 샘플 미리보기 ──')
print(train_df.head(3).to_string(index=False))

print('\n── Train 성별 분포 ──')
dist = train_df['gender'].value_counts().sort_index()
dist.index = dist.index.map({0: '남(0)', 1: '여(1)'})
print(dist.to_string())
print(f'  불균형 비율: {dist.max()/dist.min():.2f}x')

print('\n── Val 성별 분포 ──')
dist_v = val_df['gender'].value_counts().sort_index()
dist_v.index = dist_v.index.map({0: '남(0)', 1: '여(1)'})
print(dist_v.to_string())

print('\n── 발화 길이 통계 (초) ──')
print(train_df['duration'].describe().round(3))

In [ ]:
# 발화 길이 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, title) in zip(axes, [(train_df, 'Train'), (val_df, 'Val')]):
    for g, color, label in [(0, 'steelblue', '남'), (1, 'salmon', '여')]:
        ax.hist(df[df['gender']==g]['duration'], bins=50,
                alpha=0.6, color=color, label=label)
    ax.set_xlabel('발화 길이 (초)')
    ax.set_ylabel('빈도')
    ax.set_title(f'{title} 발화 길이 분포')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n짧은 발화 (< 0.5s) 제거 후 남은 샘플:')
print(f'  Train: {len(train_df):,}개')
print(f'  Val  : {len(val_df):,}개')

## 5. Mel-spectrogram 변환

### 파라미터 설명
| 파라미터 | 값 | 설명 |
|----------|-----|------|
| `SR` | 16000 | 샘플링 레이트 (16 kHz — 음성 표준) |
| `N_MELS` | 128 | Mel 필터 뱅크 수 (주파수 해상도) |
| `N_FFT` | 1024 | FFT 윈도우 크기 |
| `HOP_LENGTH` | 256 | 프레임 이동 간격 |
| `TARGET_SEC` | 3.0 | 모델 입력 고정 길이 (초) |

→ 3초 기준: 시간 프레임 ≈ 3×16000÷256 ≈ **188 frames**  
→ Mel-spectrogram shape: **(128 mel bins × 188 time frames)**  
→ CNN 입력 전 (128 × 128)으로 resize

In [ ]:
# 전처리 하이퍼파라미터
SR         = 16000
N_MELS     = 128
N_FFT      = 1024
HOP_LENGTH = 256
TARGET_SEC = 3.0    # 고정 입력 길이 (초)
IMG_SIZE   = 128    # CNN 입력 이미지 크기


def extract_melspec(wav_path, start_sec, end_sec,
                    sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                    hop_length=HOP_LENGTH, target_sec=TARGET_SEC):
    """
    WAV 파일의 [start_sec, end_sec] 구간 (단위: 초) 을
    정규화된 Mel-spectrogram (float32, [0,1])으로 변환

    - 짧은 경우: 오른쪽에 zero-padding
    - 긴 경우  : 중앙 crop
    """
    target_len = int(target_sec * sr)

    # offset/duration 은 초(s) 단위 → parse_labels에서 이미 변환됨
    y, _ = librosa.load(
        wav_path, sr=sr,
        offset=start_sec, duration=(end_sec - start_sec)
    )

    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        # 중앙 crop
        excess = len(y) - target_len
        y = y[excess // 2: excess // 2 + target_len]

    mel    = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Min-Max 정규화 [0, 1]
    mn, mx = mel_db.min(), mel_db.max()
    mel_norm = (mel_db - mn) / (mx - mn + 1e-8)

    return mel_norm.astype(np.float32)  # (N_MELS, T)

In [ ]:
# 남/여 Mel-spectrogram 시각화 (각 1개)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for g_idx, g_label in enumerate(['남(Male)', '여(Female)']):
    row = train_df[train_df['gender'] == g_idx].iloc[0]
    mel = extract_melspec(row['wav_path'], row['start'], row['end'])

    img = axes[g_idx].imshow(mel, origin='lower', aspect='auto', cmap='viridis')
    axes[g_idx].set_title(
        f'{g_label} | 원본 {row["duration"]:.2f}s → {TARGET_SEC}s 고정'
    )
    axes[g_idx].set_xlabel('Time Frame')
    axes[g_idx].set_ylabel('Mel Bin')
    plt.colorbar(img, ax=axes[g_idx])

plt.suptitle('Mel-spectrogram 예시 (dB, 정규화)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Mel-spectrogram raw shape: {mel.shape}  → resize → ({IMG_SIZE}, {IMG_SIZE})')

## 6. Dataset & DataLoader

### 데이터 증강 (학습 시만 적용)
| 기법 | 설명 |
|------|------|
| RandomHorizontalFlip | 시간 축 좌우 반전 — 성별과 무관 |
| RandomErasing | 일부 영역 마스킹 — SpecAugment 간소화 버전 |

In [ ]:
class GenderDataset(Dataset):
    _base = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),                  # (1, H, W), [0,1]
    ])
    _aug = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.08)),
    ])

    def __init__(self, df, augment=False):
        self.df        = df.reset_index(drop=True)
        self.transform = self._aug if augment else self._base

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel   = extract_melspec(row['wav_path'], row['start'], row['end'])
        # float32 [0,1] → uint8 [0,255] (PIL 변환 필요)
        mel_u8 = (mel * 255).clip(0, 255).astype(np.uint8)
        img    = self.transform(mel_u8)    # (1, IMG_SIZE, IMG_SIZE)
        label  = int(row['gender'])        # 0: 남, 1: 여
        return img, label

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_dataset = GenderDataset(train_df, augment=True)
val_dataset   = GenderDataset(val_df,   augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print(f'Train: {len(train_loader)} 배치')
print(f'Val  : {len(val_loader)} 배치')

# 배치 shape 확인
imgs, labels = next(iter(train_loader))
print(f'\n입력 shape: {imgs.shape}  (batch, channel=1, H={IMG_SIZE}, W={IMG_SIZE})')
print(f'레이블   : {labels[:8].tolist()}')
print(f'값 범위  : [{imgs.min():.3f}, {imgs.max():.3f}]')

## 7. CNN 모델 정의

### 아키텍처 (GenderCNN)
```
Input  (1, 128, 128)
  Block1: Conv(1→32)×2  + BN + ReLU + MaxPool → (32,  64, 64)
  Block2: Conv(32→64)×2 + BN + ReLU + MaxPool → (64,  32, 32)
  Block3: Conv(64→128)×2+ BN + ReLU + MaxPool → (128, 16, 16)
  Block4: Conv(128→256)×2+ BN + ReLU          → (256, 16, 16)
  AdaptiveAvgPool(4×4)                         → (256,  4,  4)
  Flatten → FC(4096→512) → ReLU → Dropout(0.5)
  FC(512→2) → 남(0) / 여(1)
```

In [ ]:
class GenderCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        def conv_block(in_ch, out_ch, pool=True):
            layers = [
                nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ]
            if pool:
                layers += [nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)]
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(1,    32),
            conv_block(32,   64),
            conv_block(64,  128),
            conv_block(128, 256, pool=False),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = GenderCNN(num_classes=2).to(device)

total_p = sum(p.numel() for p in model.parameters())
print(f'총 파라미터: {total_p:,}')

with torch.no_grad():
    dummy = torch.randn(4, 1, IMG_SIZE, IMG_SIZE).to(device)
    out   = model(dummy)
print(f'입력: {dummy.shape}  →  출력: {out.shape}')

## 8. 학습 설정

- **Loss**: CrossEntropyLoss + 클래스 불균형 보정 가중치
- **Optimizer**: AdamW (L2 정규화 내장)
- **Scheduler**: CosineAnnealingLR

In [ ]:
EPOCHS       = 30
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# 클래스 가중치: 소수 클래스에 더 높은 가중치
counts = train_df['gender'].value_counts().sort_index().values.astype(float)
class_weights = torch.tensor(1.0 / counts * counts.mean(), dtype=torch.float).to(device)
print(f'클래스 가중치 → 남(0): {class_weights[0]:.4f}, 여(1): {class_weights[1]:.4f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels  = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='  Val  ', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            preds       = out.argmax(1)
            correct    += (preds == labels).sum().item()
            n          += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / n, correct / n, all_preds, all_labels

In [ ]:
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
start_epoch  = 1

# ── 체크포인트 있으면 이어서 학습 ──────────────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    print('✅ 체크포인트 발견 → 이어서 학습합니다.')
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch  = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    history      = ckpt['history']
    print(f'   → Epoch {ckpt["epoch"]}까지 완료, Best Val Acc: {best_val_acc:.4f}')
else:
    print('체크포인트 없음 → 처음부터 학습합니다.')

print(f'\n{"Epoch":>6} {"TrLoss":>8} {"TrAcc":>7} {"ValLoss":>8} {"ValAcc":>7} {"LR":>9}')
print('─' * 55)

for epoch in range(start_epoch, EPOCHS + 1):
    tr_loss, tr_acc         = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    tag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)   # 최고 모델만 따로 저장
        tag = ' ★'

    # 매 에포크마다 체크포인트 저장 (런타임 끊겨도 이어서 학습 가능)
    torch.save({
        'epoch'       : epoch,
        'model'       : model.state_dict(),
        'optimizer'   : optimizer.state_dict(),
        'scheduler'   : scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'history'     : history,
    }, CHECKPOINT_PATH)

    cur_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6} {tr_loss:>8.4f} {tr_acc:>7.4f} {val_loss:>8.4f} {val_acc:>7.4f} {cur_lr:>9.2e}{tag}')

print(f'\n최고 Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'모델 저장 위치: {MODEL_PATH}')

## 9. 평가 및 결과 시각화

In [ ]:
# 최고 성능 모델 로드 (Drive에서)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
_, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion, device)

print(f'Best Model Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)\n')
print(classification_report(val_labels, val_preds, target_names=['남(0)', '여(1)']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
eps = range(1, len(history['train_loss']) + 1)

ax1.plot(eps, history['train_loss'], label='Train')
ax1.plot(eps, history['val_loss'],   label='Val')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss')
ax1.legend(); ax1.grid(True, alpha=0.4)

ax2.plot(eps, [a*100 for a in history['train_acc']], label='Train')
ax2.plot(eps, [a*100 for a in history['val_acc']],   label='Val')
best_ep = history['val_acc'].index(max(history['val_acc'])) + 1
ax2.axvline(best_ep, color='red', linestyle='--', alpha=0.6, label=f'Best (ep {best_ep})')
ax2.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.4)

plt.tight_layout(); plt.show()

In [ ]:
cm   = confusion_matrix(val_labels, val_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['남(Male)', '여(Female)'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix (Validation)')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')

## 10. 추론 (Inference)

추론 시 JSON에는 `startAt`, `endAt`만 있음 (speaker, gender 없음).  
→ 파싱 후 모델에 입력해 성별을 예측.

In [ ]:
def parse_inference(json_dir, wav_dir, min_duration_sec=0.5):
    """
    추론용 파싱: startAt/endAt(ms→s) 만 수집, gender=-1
    """
    records = []
    for jpath in find_files(json_dir, 'json'):
        stem     = os.path.splitext(os.path.basename(jpath))[0]
        wav_path = find_wav(wav_dir, stem)
        if wav_path is None:
            continue

        with open(jpath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        utterances = get_utterances(data) if isinstance(data, dict) else data

        for i, utt in enumerate(utterances):
            start_sec = float(utt.get('startAt', utt.get('start', 0))) / 1000.0
            end_sec   = float(utt.get('endAt',   utt.get('end',   0))) / 1000.0
            if (end_sec - start_sec) >= min_duration_sec:
                records.append({
                    'id'      : f'{stem}_{i:04d}',
                    'wav_path': wav_path,
                    'start'   : round(start_sec, 4),
                    'end'     : round(end_sec,   4),
                    'gender'  : -1,
                    'duration': round(end_sec - start_sec, 3),
                })
    return pd.DataFrame(records)


def batch_predict(df, model, device, batch_size=64):
    """DataFrame 전체에 배치 추론 → numpy array 반환"""
    dataset = GenderDataset(df, augment=False)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2)
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc='Inferring'):
            preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
    return np.array(preds)

In [ ]:
# Validation으로 추론 파이프라인 검증
preds_arr = batch_predict(val_df, model, device)
acc = accuracy_score(val_df['gender'].values, preds_arr)
print(f'Validation Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'  남(0) 예측: {(preds_arr==0).sum():,}개')
print(f'  여(1) 예측: {(preds_arr==1).sum():,}개')

In [ ]:
# ── 제출 파일 생성 ──────────────────────────────────────────────────────────
# 테스트셋 경로 (제공된 경우)
# TEST_WAV  = f'{BASE}/Test/1.원천데이터/TS_서울_구급'
# TEST_JSON = f'{BASE}/Test/2.라벨링데이터/TS_서울_구급'

# test_df    = parse_inference(TEST_JSON, TEST_WAV)
# test_preds = batch_predict(test_df, model, device)

# submission = pd.DataFrame({
#     'id'    : test_df['id'],
#     'gender': test_preds,    # 0: 남, 1: 여
# })
# submission.to_csv('/content/submission.csv', index=False)
# print(submission.head())

print('테스트 경로 설정 후 주석 해제하여 사용하세요.')

## 11. 성능 개선 팁

### 빠른 시도
| 방법 | 예상 효과 |
|------|----------|
| `TARGET_SEC` 3→5초 | 더 많은 맥락 정보 |
| `N_MELS` 128→256 | 주파수 해상도 향상 |
| Pretrained ResNet18 | Transfer Learning |
| SpecAugment (주파수/시간 마스킹) | 강력한 증강 |

### SpecAugment 추가
```python
def spec_augment(mel, freq_mask=20, time_mask=30):
    mel = mel.copy()
    f0 = np.random.randint(0, mel.shape[0] - freq_mask)
    mel[f0:f0+freq_mask, :] = 0           # 주파수 마스킹
    t0 = np.random.randint(0, mel.shape[1] - time_mask)
    mel[:, t0:t0+time_mask] = 0           # 시간 마스킹
    return mel
```

### Pretrained ResNet 사용
```python
import torchvision.models as models
resnet = models.resnet18(pretrained=True)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # 1채널 입력
resnet.fc    = nn.Linear(resnet.fc.in_features, 2)
model = resnet.to(device)
```

### 신고자 발화만 필터링 (노이즈 감소)
```python
# speaker 0을 신고자로 가정하는 경우
train_df_filtered = train_df[train_df['speaker'] == 0].copy()
```